Alright let's try this on my stupid Kaggle dataset

In [4]:
# IMPORTS

# === Section 1a. DECLARE IMPORTS ===

from importlib.metadata import version  # to verify
import logging  # for type hinting
import platform  # to verify
from typing import Final  # for type hinting

from datafun_toolkit.logger import get_logger, log_header
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

# Had to add this to test the model
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

import os
import sys
import kagglehub


# === Section 1b. CONFIGURE LOGGER ONCE PER NOTEBOOK ===

LOG: logging.Logger = get_logger("M05", level="DEBUG")
log_header(LOG, "M05")


# === Section 1c. USE THE LOGGER TO VERIFY IMPORTS ===

# If any do NOT return a version number, then that package is not installed correctly.
# Check your pyproject.toml and re-run environment setup commands.

LOG.info("Confirming installation:")
LOG.info(f"  python:       {platform.python_version()}")
LOG.info(f"  pandas:       {version('pandas')}")
LOG.info(f"  numpy:        {version('numpy')}")
LOG.info(f"  scikit-learn: {version('scikit-learn')}")
LOG.info(f"  seaborn:      {version('seaborn')}")
LOG.info(f"  matplotlib:   {version('matplotlib')}")


# === Section 1d. SET PANDAS DISPLAY CONFIGURATION (helps in notebooks) ===

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

2026-08-02 14:02:48 | INFO | M05 | === RUN START ===
2026-08-02 14:02:48 | INFO | M05 | project=M05
2026-08-02 14:02:48 | INFO | M05 | repo_dir=ml-05-ensembles
2026-08-02 14:02:48 | INFO | M05 | python=3.14.6
2026-08-02 14:02:48 | INFO | M05 | os=Windows 11
2026-08-02 14:02:48 | INFO | M05 | shell=powershell
2026-08-02 14:02:48 | INFO | M05 | cwd=notebooks
2026-08-02 14:02:48 | INFO | M05 | github_actions=False
2026-08-02 14:02:48 | INFO | M05 | Confirming installation:
2026-08-02 14:02:48 | INFO | M05 |   python:       3.14.6
2026-08-02 14:02:48 | INFO | M05 |   pandas:       3.0.3
2026-08-02 14:02:48 | INFO | M05 |   numpy:        2.5.0
2026-08-02 14:02:48 | INFO | M05 |   scikit-learn: 1.9.0
2026-08-02 14:02:48 | INFO | M05 |   seaborn:      0.13.2
2026-08-02 14:02:48 | INFO | M05 |   matplotlib:   3.11.0


In [5]:
# Download dataset and load CSV
path = kagglehub.dataset_download("anirudhchauhan/retail-store-inventory-forecasting-dataset")
df = pd.read_csv(f"{path}/retail_store_inventory.csv")

print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Loaded 73100 rows, 15 columns


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer
